In [ ]:
import pandas as pd
import numpy as np

In [58]:
data = pd.read_csv('data/ucl_matches_2021_22_to_2025_26.csv')

In [3]:
data.head(5)

,season,match_number,round_number,date_utc,location,home_team,away_team,group,home_goals,away_goals,source_url,total_goals,goal_difference_home,result_90min
0,2021-22,2,1,2021-09-14 16:45:00+00:00,Stadion Wankdorf,Young Boys,Man. United,Group F,2.0,1.0,https://fixturedownload.com/feed/json/champion...,3.0,1.0,H
1,2021-22,3,1,2021-09-14 16:45:00+00:00,Estadio Ramón Sánchez-Pizjuán,Sevilla,Salzburg,Group G,1.0,1.0,https://fixturedownload.com/feed/json/champion...,2.0,0.0,D
2,2021-22,1,1,2021-09-14 19:00:00+00:00,Estadio de la Cerámica,Villarreal,Atalanta,Group F,2.0,2.0,https://fixturedownload.com/feed/json/champion...,4.0,0.0,D
3,2021-22,4,1,2021-09-14 19:00:00+00:00,NSC Olimpiyskiy,Dynamo Kyiv,Benfica,Group E,0.0,0.0,https://fixturedownload.com/feed/json/champion...,0.0,0.0,D
4,2021-22,5,1,2021-09-14 19:00:00+00:00,Camp Nou,Barcelona,Bayern,Group E,0.0,3.0,https://fixturedownload.com/feed/json/champion...,3.0,-3.0,A


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 753 entries, 0 to 752
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   season                753 non-null    object 
 1   match_number          753 non-null    int64  
 2   round_number          753 non-null    int64  
 3   date_utc              753 non-null    object 
 4   location              753 non-null    object 
 5   home_team             753 non-null    object 
 6   away_team             753 non-null    object 
 7   group                 288 non-null    object 
 8   home_goals            752 non-null    float64
 9   away_goals            752 non-null    float64
 10  source_url            753 non-null    object 
 11  total_goals           752 non-null    float64
 12  goal_difference_home  752 non-null    float64
 13  result_90min          753 non-null    object 
dtypes: float64(4), int64(2), object(8)
memory usage: 82.5+ KB


In [5]:
data.tail(2)

,season,match_number,round_number,date_utc,location,home_team,away_team,group,home_goals,away_goals,source_url,total_goals,goal_difference_home,result_90min
751,2025-26,188,16,2026-05-06 19:00:00+00:00,Fußball Arena München,Bayern München,Paris,NaN,1.0,1.0,https://fixturedownload.com/feed/json/champion...,2.0,0.0,D
752,2025-26,189,17,2026-05-30 16:00:00+00:00,Puskás Aréna,Paris,Arsenal,NaN,NaN,NaN,https://fixturedownload.com/feed/json/champion...,NaN,NaN,D


In [59]:
data.drop(columns=['date_utc', 'source_url'], inplace=True)

In [7]:
data.tail(5)

,season,match_number,round_number,location,home_team,away_team,group,home_goals,away_goals,total_goals,goal_difference_home,result_90min
748,2025-26,185,15,Parc des Princes,Paris,Bayern München,NaN,5.0,4.0,9.0,1.0,H
749,2025-26,186,15,Estadio Metropolitano,Atleti,Arsenal,NaN,1.0,1.0,2.0,0.0,D
750,2025-26,187,16,Arsenal Stadium,Arsenal,Atleti,NaN,1.0,0.0,1.0,1.0,H
751,2025-26,188,16,Fußball Arena München,Bayern München,Paris,NaN,1.0,1.0,2.0,0.0,D
752,2025-26,189,17,Puskás Aréna,Paris,Arsenal,NaN,NaN,NaN,NaN,NaN,D


In [60]:
conditions = [data['result_90min'].eq('H'), data['result_90min'].eq('A')]
choices = [data['home_team'], data['away_team']]
data['match_winner'] = np.select(conditions, choices, default='Draw')

data[['home_team', 'away_team', 'result_90min', 'match_winner']].head(10)

,home_team,away_team,result_90min,match_winner
0,Young Boys,Man. United,H,Young Boys
1,Sevilla,Salzburg,D,Draw
2,Villarreal,Atalanta,D,Draw
3,Dynamo Kyiv,Benfica,D,Draw
4,Barcelona,Bayern,A,Bayern
5,LOSC,Wolfsburg,D,Draw
6,Malmö,Juventus,A,Juventus
7,Chelsea,Zenit,H,Chelsea
8,Besiktas,Dortmund,A,Dortmund
9,Sheriff,Shakhtar Donetsk,H,Sheriff


In [61]:
data['knockout_aggregate'] = None
data['knockout_winner'] = None


def tie_id(row):
    team_1, team_2 = sorted([row['home_team'], row['away_team']])
    return f"{row['season']}|{team_1}|{team_2}"

knockout_matches = data[
    data['group'].isna() & data['home_goals'].notna() & data['away_goals'].notna()
].copy().sort_values(['season', 'round_number', 'match_number'])
processed_ties = set()

for _, match1 in knockout_matches.iterrows():
    teams_key = tuple(sorted([match1['home_team'], match1['away_team']]))
    tie_key = (match1['season'], teams_key)
    if tie_key in processed_ties:
        continue

    match2_candidates = knockout_matches[
        (knockout_matches['season'] == match1['season'])
        & (
            ((knockout_matches['home_team'] == match1['home_team']) & (knockout_matches['away_team'] == match1['away_team']))
            | ((knockout_matches['home_team'] == match1['away_team']) & (knockout_matches['away_team'] == match1['home_team']))
        )
        & (knockout_matches['match_number'] != match1['match_number'])
        & (knockout_matches['round_number'].sub(match1['round_number']).abs() == 1)
    ]

    if len(match2_candidates) != 1:
        continue

    match2 = match2_candidates.iloc[0]
    leg1, leg2 = (match1, match2) if match1['match_number'] < match2['match_number'] else (match2, match1)
    team_a, team_b = leg1['home_team'], leg1['away_team']

    leg1_score = f"{int(leg1['home_goals'])}-{int(leg1['away_goals'])}"
    team_a_goals = leg1['home_goals'] + leg2['away_goals']
    team_b_goals = leg1['away_goals'] + leg2['home_goals']
    aggregate = f"{int(team_a_goals)}-{int(team_b_goals)}"

    if team_a_goals > team_b_goals:
        winner = team_a
    elif team_b_goals > team_a_goals:
        winner = team_b
    else:
        team_a_away = leg2['away_goals']
        team_b_away = leg1['away_goals']
        if team_a_away > team_b_away:
            winner = team_a
        elif team_b_away > team_a_away:
            winner = team_b
        else:
            winner = None

    data.loc[data['match_number'] == leg1['match_number'], 'knockout_aggregate'] = leg1_score
    data.loc[data['match_number'] == leg2['match_number'], 'knockout_aggregate'] = aggregate
    data.loc[
        (data['match_number'] == leg1['match_number']) | (data['match_number'] == leg2['match_number']),
        'knockout_winner'
    ] = winner
    processed_ties.add(tie_key)

data[data['group'].isna()][['season', 'round_number', 'home_team', 'away_team', 'knockout_aggregate', 'knockout_winner']].tail(10)

,season,round_number,home_team,away_team,knockout_aggregate,knockout_winner
743,2025-26,13,Barcelona,Atleti,0-2,Atleti
744,2025-26,14,Atleti,Barcelona,2-3,Atleti
745,2025-26,14,Liverpool,Paris,4-0,Paris
746,2025-26,14,Arsenal,Sporting CP,0-1,Arsenal
747,2025-26,14,Bayern München,Real Madrid,4-6,Bayern München
748,2025-26,15,Paris,Bayern München,5-4,Paris
749,2025-26,15,Atleti,Arsenal,1-1,Arsenal
750,2025-26,16,Arsenal,Atleti,1-2,Arsenal
751,2025-26,16,Bayern München,Paris,6-5,Paris
752,2025-26,17,Paris,Arsenal,None,None


In [64]:
data.tail(5)

,season,match_number,round_number,home_team,away_team,group,home_goals,away_goals,goal_difference_home,result_90min,match_winner,knockout_aggregate,knockout_winner
748,2025-26,185,15,Paris,Bayern München,NaN,5.0,4.0,1.0,H,Paris,5-4,Paris
749,2025-26,186,15,Atleti,Arsenal,NaN,1.0,1.0,0.0,D,Draw,1-1,Arsenal
750,2025-26,187,16,Arsenal,Atleti,NaN,1.0,0.0,1.0,H,Arsenal,1-2,Arsenal
751,2025-26,188,16,Bayern München,Paris,NaN,1.0,1.0,0.0,D,Draw,6-5,Paris
752,2025-26,189,17,Paris,Arsenal,NaN,NaN,NaN,NaN,D,Draw,None,None


In [63]:
data.drop(columns=['total_goals', 'location'], inplace=True)

In [65]:
data.to_csv('data/dataset_cleaned.csv', index=False)